# D4-0b: where does adaptive allocation pay at equal compute? (validation-only design study)

Follow-up to D4-0 (`docs/research-notes/2026-09-29-d4-0-adaptive-time-stepping.md`), where plain uniform refinement beat every adaptive policy at equal total compute and D4-1 stayed closed. **This notebook is an exploratory design study, not a gate.** It uses only a tuning family and a validation family; **the test family (seed 2002) is never generated or read here.** It runs on CPU.

**What changed since D4-0, and why.**

- **Pass semantics.** D4-0 applied one refinement per scoring pass, and charged every refinement its own re-solve. That penalises every policy, and it charged the uniform baseline `O(n²)` for a pass a real solver does in `O(n)`. Here a *pass* scores once (adaptive policies), refines a whole set of intervals and re-solves once from the leftmost refined interval (`n + k − j_min` CN steps for `k` refinements).
- **Policies** (all deployable): `uniform_pass` (refine every interval of the current maximum length; no scoring), and Dörfler marking (refine the smallest set of intervals whose score covers a fraction `θ` of the total) on three scores that share the same step-doubling estimate `τ̂`: `residual` (`‖τ̂‖`), `goal_local` (`|cᵀτ̂|`) and `adjoint` (`|Λᵀτ̂|`, the discrete co-state, computed exactly from the known model). Scoring is charged per pass: `2n` CN steps (`3n` with the co-state sweep). `θ ∈ {0.3, 0.5, 0.7, 0.85, 0.95}` is chosen per policy and family on the tuning family, with the same grid for every policy.
- **Work–precision measure.** For each instance, the compute a policy needs to close a fraction of the gap between the initial objective and the finest-grid objective: targets at 10 %, 3 % and 1 % of the gap remaining. This does not depend on how coarse a pass is. A target not reached within the cap (2 × the finest grid's steps) counts as the cap, which understates that policy's compute and is conservative against its rival.
- **Localisation sweep.** Three families with pulse widths of 0.01–0.04 (`smooth`, the D4-0 family), 0.001–0.004 (`sharp`) and 0.0001–0.0004 (`sharper`), with amplitude scaled so the forcing impulses stay comparable. Theory says adaptivity pays only when the features are localised enough to outweigh the scoring overhead. The `sharp` and `sharper` families were added *because* of that, after D4-0 showed no payoff on `smooth`; they are a design choice motivated by theory, and a result here says where adaptivity pays, not that it pays in general.
- **Scoring-price what-ifs.** Because decisions never depend on costs, the same traces are re-priced with the scoring cost scaled by 1 (the real ledger), 0.25 and 0. **Scales below 1 are hypothetical.** A scale `s` is equivalent to solves being `1/s` times as expensive relative to scoring, which is the regime a learned, amortised scorer is meant for; scale 0 shows what allocation quality alone would buy.

**Design history (disclosed).** A first execution of this notebook, at commit `10c0557`, used the grid `θ ∈ {0.3, 0.5, 0.7}`. The tuning table showed **`θ = 0.7`, the top of the grid, selected for all nine (family, policy) pairs, with the cost still falling as `θ` rose**, so the adaptive policies had been tuned against a truncated grid. The grid was extended to 0.85 and 0.95 and nothing else was changed. That decision rests on the tuning-family table alone, but **the validation results of the first execution had already been printed** and were seen before the change. The first execution is kept as a superseded record. The selection is checked for a boundary optimum again in this notebook and reported.

**Frozen decision rule** (stated before any validation number is read). A cell `(family, scoring price)` is a *candidate regime for D4-1* if, on the validation family:

1. the co-state marking policy needs less compute than `uniform_pass`: the CI of `mean(log(compute_adjoint / compute_uniform))` lies below 0 at two adjacent targets; **and**
2. the co-state policy needs less compute than both `residual` and `goal_local` marking, by the same criterion.

A D4-1 design must fix its regime from these cells and be gated separately, with the test family read once. A cell that meets (1) but not (2) says adaptivity pays but the co-state adds nothing there; a cell that meets neither says D4 gives no compute-adjusted opportunity in that regime.

In [ ]:
# ================================================================
# 0. Package (CPU is enough)
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
try:
    from adjointrwm.domains import marking_policy, sample_family, uniform_pass_policy  # noqa: F401
except ImportError as error:
    raise RuntimeError(f'{REPO_REF!r} predates the D4-0b modules; set REPO_REF to a commit that has them.') from error

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'
print('adjointrwm at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '', '| runs ->', RUN_ROOT)

In [ ]:
# ================================================================
# 1. Frozen configuration and immutable run directory
# ================================================================
import dataclasses
import datetime
import json
import platform
import time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from adjointrwm.domains import AdaptiveTimeSteppingDomain, FAMILIES, sample_family
from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_json


@dataclass(frozen=True)
class D4bConfig:
    families: tuple = ('smooth', 'sharp', 'sharper')
    tuning_seed: int = 3003           # chooses theta only
    validation_seed: int = 1001       # reported; the D4-0 validation family
    tuning_instances: int = 10
    validation_instances: int = 20
    # The test family (seed 2002) is deliberately never generated or read in this notebook.
    thetas: tuple = (0.3, 0.5, 0.7, 0.85, 0.95)
    kinds: tuple = ('residual', 'goal_local', 'adjoint')
    gap_fractions: tuple = (0.1, 0.03, 0.01)      # fraction of the (initial - finest-grid) gap that may remain
    cap_in_finest_grids: float = 2.0              # compute cap = this * (finest grid steps)
    max_passes: int = 40
    decision_scales: tuple = (1.0, 0.25, 0.0)     # 1 = the real ledger; the others are what-ifs
    bootstrap_resamples: int = 10000


CFG = D4bConfig()
CONFIG_DICT = asdict(CFG)
CONFIG_HASH = sha256_json(CONFIG_DICT)
RUN_ID = 'd4_0b_adaptivity_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
PATHS = {k: RUN_DIR / k for k in ('config', 'artifacts', 'figures', 'reports', 'tests')}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

DOMAIN = AdaptiveTimeSteppingDomain(objective_kind='bound')
FAMILY_TABLE = {name: {'pulse_width': FAMILIES[name][0], 'amplitude_scale': FAMILIES[name][1], 'max_depth': FAMILIES[name][2]}
                for name in CFG.families}
INSTANCES = {(name, split): sample_family(name, seed, count)
             for name in CFG.families
             for split, seed, count in (('tuning', CFG.tuning_seed, CFG.tuning_instances),
                                        ('validation', CFG.validation_seed, CFG.validation_instances))}
MANIFEST = {f'{name}/{split}': [asdict(i) for i in insts] for (name, split), insts in INSTANCES.items()}
atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT, 'repo_dirty': REPO_DIRTY,
    'notebook': 'notebooks/04-domains/d4_0b_where_adaptivity_pays.ipynb', 'families': FAMILY_TABLE,
    'domain_spec': DOMAIN.spec.to_dict(), 'domain_spec_sha256': DOMAIN.spec.checksum(),
    'test_family_read': False,
})
atomic_write_json(PATHS['config'] / 'instances.json', {**MANIFEST, 'sha256': sha256_json(MANIFEST)})
atomic_write_json(PATHS['config'] / 'environment.json', {
    'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'hardware': 'CPU (timings are not a systems claim)',
})
print('Run:', RUN_ID, '| config', CONFIG_HASH[:12])

In [ ]:
# ================================================================
# 2. Correctness of the batch layer on non-uniform grids (before anything else is read)
# ================================================================
from adjointrwm.domains import LinearODEInstance
from adjointrwm.domains.linear_ode import reference_solution

rows = []
for name in CFG.families:
    for inst in INSTANCES[(name, 'tuning')][:3]:
        state = DOMAIN.initial_state(inst)
        picked = [DOMAIN.legal_candidates(state)[j] for j in (1, 5, 6)]
        batch = DOMAIN.apply_batch(state, picked)
        sequential = state
        for candidate in picked:
            sequential = DOMAIN.apply(sequential, candidate)
        weighted = DOMAIN.weighted_local_errors(batch, inst).sum()
        rows.append({'family': name, 'instance': inst.name,
                     'batch_equals_sequential_nodes': batch.nodes == sequential.nodes,
                     'batch_vs_sequential_max_abs_diff': float(np.abs(batch.y - sequential.y).max()),
                     'representation_abs_diff': float(abs(weighted - DOMAIN.signed_error(batch, inst))),
                     'abs_qoi_error': abs(DOMAIN.signed_error(batch, inst))})
checks = pd.DataFrame(rows)
convergence = []
for name in CFG.families:
    inst = INSTANCES[(name, 'tuning')][0]
    finer = LinearODEInstance(**{**inst.__dict__, 'max_depth': inst.max_depth + 1})
    convergence.append({'family': name, 'instance': inst.name,
                        'max_abs_diff_yT': float(np.abs(reference_solution(inst)[-1] - reference_solution(finer)[-1]).max())})
convergence = pd.DataFrame(convergence)
CORRECTNESS = {
    'batch_equals_sequential': bool(checks['batch_equals_sequential_nodes'].all() and (checks['batch_vs_sequential_max_abs_diff'] < 1e-10).all()),
    'error_representation_on_batch_grids': bool((checks['representation_abs_diff'] <= 1e-9 * np.maximum(1.0, checks['abs_qoi_error'])).all()),
    'reference_converged': bool((convergence['max_abs_diff_yT'] < 1e-8).all()),
    'max_representation_abs_diff': float(checks['representation_abs_diff'].max()),
    'max_reference_diff': float(convergence['max_abs_diff_yT'].max()),
}
CORRECTNESS['passed'] = bool(CORRECTNESS['batch_equals_sequential'] and CORRECTNESS['error_representation_on_batch_grids'] and CORRECTNESS['reference_converged'])
atomic_write_json(PATHS['tests'] / 'correctness.json', CORRECTNESS)
checks.to_csv(PATHS['tests'] / 'batch_checks.csv', index=False)
convergence.to_csv(PATHS['tests'] / 'reference_convergence.csv', index=False)
print(json.dumps(CORRECTNESS, indent=2))
if not CORRECTNESS['passed']:
    raise RuntimeError('Correctness failed: nothing below may be read. See tests/.')

In [ ]:
# ================================================================
# 3. Choose theta on the tuning family (same grid for every policy)
# ================================================================
from adjointrwm.domains import evaluate_work_precision, marking_policy, uniform_pass_policy


def target_fn(domain, instance):
    floor = domain.finest_objective(instance)                   # evaluation only (privileged)
    start = domain.objective(domain.initial_state(instance), instance)
    return {f'gap{int(round(100 * phi))}%': floor + phi * (start - floor) for phi in CFG.gap_fractions}


TARGETS = [f'gap{int(round(100 * phi))}%' for phi in CFG.gap_fractions]   # ordered loosest -> tightest


def cap_for(instance):
    return CFG.cap_in_finest_grids * instance.n_fine


def run_family(name, split, policies, scales):
    frames = []
    for inst in INSTANCES[(name, split)]:
        frames.append(evaluate_work_precision(DOMAIN, [inst], policies, target_fn, compute_cap=cap_for(inst),
                                              max_steps=CFG.max_passes, random_draws=1, decision_scales=scales))
    return pd.concat(frames, ignore_index=True)


started = time.perf_counter()
TUNING_ROWS, SELECTED_THETA = [], {}
for name in CFG.families:
    grid = [marking_policy(kind, theta) for kind in CFG.kinds for theta in CFG.thetas]
    frame = run_family(name, 'tuning', grid, (1.0,))
    frame['cap'] = frame['instance'].map({i.name: cap_for(i) for i in INSTANCES[(name, 'tuning')]})
    frame['compute'] = np.minimum(frame['compute'], frame['cap'])
    score = frame.groupby('policy')['compute'].apply(lambda c: float(np.exp(np.log(c).mean())))   # geometric mean over instances and targets
    for kind in CFG.kinds:
        options = {t: score[f'mark_{kind}_{t:g}'] for t in CFG.thetas}
        SELECTED_THETA[(name, kind)] = min(options, key=options.get)
        TUNING_ROWS.append({'family': name, 'kind': kind, **{f'theta_{t:g}': options[t] for t in CFG.thetas},
                            'selected': SELECTED_THETA[(name, kind)],
                            'selected_at_grid_edge': SELECTED_THETA[(name, kind)] in (min(CFG.thetas), max(CFG.thetas))})
print(f'tuning done in {time.perf_counter() - started:.0f}s')
TUNING = pd.DataFrame(TUNING_ROWS)
TUNING.to_csv(PATHS['artifacts'] / 'theta_tuning.csv', index=False)
print(TUNING.to_string(index=False))

In [ ]:
# ================================================================
# 4. Validation family: compute to reach each target, at every scoring price
# ================================================================
from adjointrwm.domains import work_precision_summary

started = time.perf_counter()
FRAMES, SUMMARY = {}, {}
PAIRS = [('adjoint', 'uniform_pass'), ('residual', 'uniform_pass'), ('goal_local', 'uniform_pass'),
         ('adjoint', 'residual'), ('adjoint', 'goal_local')]
for name in CFG.families:
    policies = [uniform_pass_policy()] + [dataclasses.replace(marking_policy(kind, SELECTED_THETA[(name, kind)]), name=kind) for kind in CFG.kinds]
    frame = run_family(name, 'validation', policies, CFG.decision_scales)
    frame.insert(0, 'family', name)
    FRAMES[name] = frame
    cap = float(cap_for(INSTANCES[(name, 'validation')][0]))   # identical for all instances of a family
    SUMMARY[name] = work_precision_summary(frame.drop(columns='family'), PAIRS, compute_cap=cap, num_resamples=CFG.bootstrap_resamples)
    print(f'{name}: {time.perf_counter() - started:.0f}s elapsed')
ALL = pd.concat(FRAMES.values(), ignore_index=True)
ALL.to_parquet(PATHS['artifacts'] / 'validation_work_precision.parquet', index=False)
atomic_write_json(PATHS['artifacts'] / 'validation_summary.json', {
    'selected_theta': {f'{n}/{k}': v for (n, k), v in SELECTED_THETA.items()}, 'pairs': PAIRS, 'summary': SUMMARY,
    'note': 'decision scales other than 1 are hypothetical re-pricings of the same traces',
})
for name in CFG.families:
    for scale in sorted(SUMMARY[name], key=float, reverse=True):
        for target in TARGETS:
            e = SUMMARY[name][scale][target]
            d = e['differences']
            print(f"{name:8s} scale {scale:>4s} {target:6s} median compute " +
                  ' '.join(f"{p}={e['median_compute'][p]:.0f}" for p in ('uniform_pass', 'residual', 'goal_local', 'adjoint')) +
                  f" | adjoint/uniform {d['adjoint / uniform_pass']['ratio_of_geometric_means']:.2f} ({d['adjoint / uniform_pass']['reading']})")

In [ ]:
# ================================================================
# 5. Frozen decision rule, table and figure
# ================================================================
def two_adjacent(flags):
    return any(flags[i] and flags[i + 1] for i in range(len(flags) - 1))


CELLS = []
for name in CFG.families:
    for scale in sorted(SUMMARY[name], key=float, reverse=True):
        block = SUMMARY[name][scale]
        pays = [block[t]['differences']['adjoint / uniform_pass']['ci_high'] < 0 for t in TARGETS]
        beats_residual = [block[t]['differences']['adjoint / residual']['ci_high'] < 0 for t in TARGETS]
        beats_goal_local = [block[t]['differences']['adjoint / goal_local']['ci_high'] < 0 for t in TARGETS]
        others_pay = {kind: two_adjacent([block[t]['differences'][f'{kind} / uniform_pass']['ci_high'] < 0 for t in TARGETS])
                      for kind in ('residual', 'goal_local')}
        rule1, rule2 = two_adjacent(pays), two_adjacent(beats_residual) and two_adjacent(beats_goal_local)
        widths = [p.width for i in INSTANCES[(name, 'validation')] for p in i.pulses]
        CELLS.append({'family': name, 'median_pulse_width': float(np.median(widths)), 'scoring_price_scale': float(scale),
                      'adjoint_beats_uniform_two_adjacent_targets': rule1,
                      'adjoint_beats_residual_and_goal_local_two_adjacent_targets': rule2,
                      'residual_beats_uniform_two_adjacent_targets': others_pay['residual'],
                      'goal_local_beats_uniform_two_adjacent_targets': others_pay['goal_local'],
                      'candidate_regime_for_d4_1': bool(rule1 and rule2),
                      **{f'ratio_adjoint_over_uniform_{t}': block[t]['differences']['adjoint / uniform_pass']['ratio_of_geometric_means'] for t in TARGETS},
                      **{f'ratio_residual_over_uniform_{t}': block[t]['differences']['residual / uniform_pass']['ratio_of_geometric_means'] for t in TARGETS},
                      **{f'ratio_adjoint_over_residual_{t}': block[t]['differences']['adjoint / residual']['ratio_of_geometric_means'] for t in TARGETS}})
CELLS = pd.DataFrame(CELLS)
CELLS.to_csv(PATHS['figures'] / 'phase_table.csv', index=False)
print(CELLS[['family', 'scoring_price_scale', 'adjoint_beats_uniform_two_adjacent_targets', 'residual_beats_uniform_two_adjacent_targets',
             'adjoint_beats_residual_and_goal_local_two_adjacent_targets', 'candidate_regime_for_d4_1']].to_string(index=False))

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(CFG.decision_scales), figsize=(4.3 * len(CFG.decision_scales), 4.2), sharey=True)
for ax, scale in zip(np.atleast_1d(axes), CFG.decision_scales):
    block = CELLS[CELLS['scoring_price_scale'] == scale].sort_values('median_pulse_width', ascending=False)
    for kind in ('adjoint', 'residual'):
        ax.plot(block['median_pulse_width'], block[f'ratio_{kind}_over_uniform_gap3%'], marker='o', label=kind)
    ax.axhline(1.0, color='grey', lw=1)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.invert_xaxis()
    ax.set_xlabel('median pulse width (smaller = more localised)')
    ax.set_title(f"scoring price x{scale:g}" + ('' if scale == 1 else ' (what-if)'))
np.atleast_1d(axes)[0].set_ylabel('compute vs uniform_pass to close 97 % of the gap\n(below 1: adaptive is cheaper)')
np.atleast_1d(axes)[0].legend()
fig.suptitle(f'{RUN_ID}: validation family, geometric-mean compute ratio', fontsize=10)
fig.savefig(PATHS['figures'] / 'phase_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

REPORT = {
    'run_id': RUN_ID, 'milestone': 'D4-0b (exploratory design study)', 'config_hash': CONFIG_HASH, 'repo_commit': REPO_COMMIT,
    'correctness': CORRECTNESS, 'selected_theta': {f'{n}/{k}': v for (n, k), v in SELECTED_THETA.items()},
    'theta_selected_at_grid_edge': {f'{n}/{k}': bool(v in (min(CFG.thetas), max(CFG.thetas))) for (n, k), v in SELECTED_THETA.items()},
    'cells': CELLS.to_dict(orient='records'),
    'candidate_regimes_for_d4_1': CELLS[CELLS['candidate_regime_for_d4_1']][['family', 'scoring_price_scale']].to_dict(orient='records'),
    'test_family_read': False,
    'claim_boundary': ('Exploratory design study on a tuning and a validation family of the analytic D4 benchmark; the test family was not '
                       'read. Scoring-price scales below 1 are hypothetical re-pricings. No learned arm exists, so no H2 statement follows. '
                       'The sharp and sharper families were added after D4-0 to vary localisation.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# D4-0b `{RUN_ID}` (exploratory)', '', f"Correctness: **{'PASS' if CORRECTNESS['passed'] else 'FAIL'}** · "
         f"candidate regimes for D4-1: **{len(REPORT['candidate_regimes_for_d4_1'])}** of {len(CELLS)} cells", '',
         '| Family | Median pulse width | Scoring price | Adjoint beats uniform | Adjoint beats residual and goal-local | Candidate for D4-1 |', '|---|---:|---:|---|---|---|']
for _, r in CELLS.iterrows():
    lines.append(f"| {r['family']} | {r['median_pulse_width']:.5f} | x{r['scoring_price_scale']:g} | {r['adjoint_beats_uniform_two_adjacent_targets']} | "
                 f"{r['adjoint_beats_residual_and_goal_local_two_adjacent_targets']} | {r['candidate_regime_for_d4_1']} |")
lines += ['', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

| Cell outcome | Meaning | Next step |
|---|---|---|
| Adjoint beats uniform, and beats residual and goal-local | Adaptivity pays there, and the co-state adds value | A candidate regime for D4-1 (learned direct critic vs co-state-featured critic), gated separately |
| Adjoint beats uniform, but not both other adaptive policies | Adaptivity pays; the co-state's extra sweep does not | The co-state is not the source of the win in that regime |
| Residual or goal-local beats uniform, adjoint does not | The co-state sweep costs more than it saves | Regime is a candidate only for cheaper scorers |
| Nothing beats uniform | No compute-adjusted opportunity there | Report it; do not tune a family until it does |
| A win only at price ×0.25 or ×0 | Adaptivity pays only if scoring is much cheaper than exact scoring | The bar a learned scorer must clear: measured cost at most that fraction of an exact scoring pass |

Import with the `import-run` skill, write the note with `research-note`, and run `claim-check`.